# Steam 인디 게임 시장 전체 추정을 위한 층화 추출

## 목적
- 샘플링 프레임: `steam_indie_list.csv` (2023~2025년 출시 게임)
- 층화 축 1: **규모** — `owners_lower` 기준 (large / mid / small)
- 층화 축 2: **리뷰 신뢰도** — Wilson Score 기준 (high / mid / low)
- 배분 방식: **비례 배분 + 최소 하한선** (모집단 대표성 확보)

## 제외 대상 (별도 분석)
| 제외 대상 | 이유 |
|---|---|
| Early Access | 출시일 기준 모호, 리뷰 누적 패턴 상이 |
| F2P | `owners_lower`로 흥행 측정 불가, 리뷰 패턴 상이 |

## 층화 변수 기준
| 축 | 층 | 기준 |
|---|---|---|
| 규모 | large | owners_lower ≥ 200,000 |
| 규모 | mid | 20,000 ≤ owners_lower < 200,000 |
| 규모 | small | owners_lower < 20,000 |
| 신뢰도 | high | total_reviews ≥ 381 (긍정률 오차 ±5% 이하) |
| 신뢰도 | mid | 39 ≤ total_reviews < 381 (±5~15%) |
| 신뢰도 | low | 10 ≤ total_reviews < 39 (±15% 초과) |

## 1. 라이브러리 로드

In [10]:
import pandas as pd
import numpy as np
import ast
import warnings
warnings.filterwarnings('ignore')

# ── 설정 ────────────────────────────────────────────
RANDOM_SEED     = 42
MIN_REVIEWS     = 10     # 분석 가능 최소 리뷰 수
TOTAL_N         = 200    # 목표 표본 수
MIN_PER_STRATUM = 15     # 층당 최소 추출 수 (하한선)

# 규모 기준 (owners_lower)
LARGE_THRESHOLD = 200_000
MID_THRESHOLD   =  20_000

np.random.seed(RANDOM_SEED)
print('완료')

완료


## 2. 데이터 로드 및 전처리

In [11]:
df = pd.read_csv('../../../data/raw/steam_indie_list.csv')

df['total_reviews'] = df['positive'] + df['negative']
df['release_date']  = pd.to_datetime(df['release_date'], errors='coerce')

def parse_owners_lower(s):
    try:
        return int(s.split('..')[0].strip().replace(',', ''))
    except:
        return 0

def parse_genres(g):
    try:
        return ast.literal_eval(g)
    except:
        return []

df['owners_lower']    = df['owners'].apply(parse_owners_lower)
df['genres_list']     = df['genres'].apply(parse_genres)
df['is_f2p']          = df['genres_list'].apply(lambda gl: 'Free To Play' in gl)
df['is_early_access'] = df['genres_list'].apply(lambda gl: 'Early Access' in gl)

print(f'원본: {len(df):,}개')
print(f'Early Access: {df["is_early_access"].sum():,}개 ({df["is_early_access"].mean():.1%})')
print(f'F2P: {df["is_f2p"].sum():,}개 ({df["is_f2p"].mean():.1%})')

원본: 61,266개
Early Access: 6,340개 (10.3%)
F2P: 3,446개 (5.6%)


## 3. 필터 적용

**출시연도: 2023 ~ 2025년** — 최근 인디 게임 시장 추세를 반영.  
**Early Access / F2P 제외** — 흥행 지표(`owners_lower`) 및 리뷰 패턴이 달라 별도 분석 대상으로 분리.

In [12]:
df_ea  = df[df['is_early_access']].copy()           # Early Access 별도 보관
df_f2p = df[~df['is_early_access'] & df['is_f2p']].copy()  # F2P 별도 보관
df_f   = df[
    (df['total_reviews'] >= MIN_REVIEWS) &
    (df['release_date'].dt.year >= 2023) &
    (df['release_date'].dt.year <= 2025) &
    (~df['is_early_access']) &
    (~df['is_f2p'])
].copy()

print(f'전체              : {len(df):,}개')
print(f'Early Access 제외 : {len(df_ea):,}개 → 별도 분석')
print(f'F2P 제외          : {len(df_f2p):,}개 → 별도 분석')
print(f'메인 모집단       : {len(df_f):,}개  (2023~2025년, 리뷰 {MIN_REVIEWS}개 이상, EA·F2P 제외)')
print(f'\n출시연도 분포 (메인):')
print(df_f['release_date'].dt.year.value_counts().sort_index().to_string())

전체              : 61,266개
Early Access 제외 : 6,340개 → 별도 분석
F2P 제외          : 3,064개 → 별도 분석
메인 모집단       : 9,692개  (2023~2025년, 리뷰 10개 이상, EA·F2P 제외)

출시연도 분포 (메인):
release_date
2023    3498
2024    4180
2025    2014


## 4. Wilson Score 경계값 계산

리뷰 신뢰도 층 경계를 데이터 기반으로 산출한다.  
긍정률 95% 신뢰 구간의 최대 오차(margin)가 기준값 이하가 되는 최소 리뷰 수를 구한다.

In [13]:
Z = 1.96

def wilson_margin(n):
    """n개 리뷰일 때 긍정률 95% CI 최대 오차 (worst case: p=0.5)"""
    p     = 0.5
    denom = 1 + Z**2 / n
    return (Z / denom) * np.sqrt(p*(1-p)/n + Z**2/(4*n**2)) * 100

def find_n_for_margin(target_pct):
    for n in range(1, 10000):
        if wilson_margin(n) <= target_pct:
            return n
    return 10000

LOW_BOUNDARY  = find_n_for_margin(15)  # low → mid 경계
HIGH_BOUNDARY = find_n_for_margin(5)   # mid → high 경계

print(f'low  (±15% 초과): 리뷰 {MIN_REVIEWS} ~ {LOW_BOUNDARY - 1}개')
print(f'mid  (±5~15%)   : 리뷰 {LOW_BOUNDARY} ~ {HIGH_BOUNDARY - 1}개')
print(f'high (±5% 이하) : 리뷰 {HIGH_BOUNDARY}개 이상')

low  (±15% 초과): 리뷰 10 ~ 38개
mid  (±5~15%)   : 리뷰 39 ~ 380개
high (±5% 이하) : 리뷰 381개 이상


## 5. 층 할당

In [14]:
def assign_stratum(row):
    scale = (
        'large' if row['owners_lower'] >= LARGE_THRESHOLD else
        'mid'   if row['owners_lower'] >= MID_THRESHOLD   else
        'small'
    )
    trust = (
        'high' if row['total_reviews'] >= HIGH_BOUNDARY else
        'mid'  if row['total_reviews'] >= LOW_BOUNDARY  else
        'low'
    )
    return f'{scale}_{trust}'

df_f['stratum'] = df_f.apply(assign_stratum, axis=1)

pop = df_f['stratum'].value_counts().sort_index()
N   = len(df_f)

print(f'모집단: {N:,}개\n')
print(f'{"층":<14} {"게임 수":>8}  {"비중":>7}')
print('-' * 35)
for stratum, cnt in pop.items():
    print(f'{stratum:<14} {cnt:>8,}  {cnt/N*100:>6.1f}%')
print('-' * 35)
print(f'{"합계":<14} {N:>8,}  {100.0:>6.1f}%')

모집단: 9,692개

층                  게임 수       비중
-----------------------------------
large_high          338     3.5%
large_low            16     0.2%
large_mid            16     0.2%
mid_high            914     9.4%
mid_low             483     5.0%
mid_mid             697     7.2%
small_high          172     1.8%
small_low         4,195    43.3%
small_mid         2,861    29.5%
-----------------------------------
합계                9,692   100.0%


## 6. 표본 배분 — 비례 배분 + 최소 하한선

순수 비례 배분은 small_low 같은 대형 층에 표본이 과도하게 몰리고,  
large_low 같은 소형 층이 통계적으로 의미 없는 숫자가 된다.  
이를 보완하기 위해 **최소 하한선**을 적용하고, 총합이 TOTAL_N을 초과하면 큰 층에서 비례 축소한다.

In [15]:
def compute_sample_plan(pop_series, total_n, min_floor):
    # 1단계: 비례 배분
    proportional = (pop_series / pop_series.sum() * total_n).round().astype(int)

    # 2단계: 하한 적용 + 모집단 상한
    floored = proportional.clip(lower=min_floor)
    floored = floored.combine(pop_series, min)  # 모집단 초과 방지

    # 3단계: 초과분을 하한 초과 층에서 비례 축소
    overflow = floored.sum() - total_n
    if overflow > 0:
        reducible = floored[(floored > min_floor) & (floored < pop_series)]
        if len(reducible) > 0:
            above = reducible - min_floor
            cut   = (above / above.sum() * overflow).round().astype(int)
            # 반올림 오차 보정
            diff = cut.sum() - overflow
            if diff != 0:
                cut.iloc[cut.argmax()] -= diff
            floored[reducible.index] -= cut

    return floored


sample_plan = compute_sample_plan(pop, TOTAL_N, MIN_PER_STRATUM)

proportional = (pop / pop.sum() * TOTAL_N).round().astype(int)

print(f'목표 표본: {TOTAL_N}개  |  층당 최소: {MIN_PER_STRATUM}개\n')
print(f'{"층":<14} {"모집단":>8}  {"비중":>7}  {"비례":>6}  {"최종":>6}  {"추출률":>7}  비고')
print('-' * 68)
for stratum in pop.index:
    cnt  = pop[stratum]
    prop = proportional[stratum]
    n    = sample_plan[stratum]
    rate = n / cnt * 100
    note = '전수' if n == cnt else ('하한' if n == MIN_PER_STRATUM and prop < MIN_PER_STRATUM else '')
    print(f'{stratum:<14} {cnt:>8,}  {cnt/N*100:>6.1f}%  {prop:>6}  {n:>6}  {rate:>6.1f}%  {note}')
print('-' * 68)
print(f'{"합계":<14} {N:>8,}  {"100.0%":>7}  {proportional.sum():>6}  {sample_plan.sum():>6}')

SAMPLE_PLAN = sample_plan.to_dict()

목표 표본: 200개  |  층당 최소: 15개

층                   모집단       비중      비례      최종      추출률  비고
--------------------------------------------------------------------
large_high          338     3.5%       7      15     4.4%  하한
large_low            16     0.2%       0      15    93.8%  하한
large_mid            16     0.2%       0      15    93.8%  하한
mid_high            914     9.4%      19      17     1.9%  
mid_low             483     5.0%      10      15     3.1%  하한
mid_mid             697     7.2%      14      15     2.2%  하한
small_high          172     1.8%       4      15     8.7%  하한
small_low         4,195    43.3%      87      54     1.3%  
small_mid         2,861    29.5%      59      39     1.4%  
--------------------------------------------------------------------
합계                9,692   100.0%     200     200


## 7. 층화 추출

In [16]:
sampled_frames = []

for stratum, n in SAMPLE_PLAN.items():
    pool     = df_f[df_f['stratum'] == stratum]
    actual_n = min(n, len(pool))
    sample   = pool.sample(n=actual_n, random_state=RANDOM_SEED)
    sampled_frames.append(sample)

df_sample = pd.concat(sampled_frames).reset_index(drop=True)
print(f'추출 완료: {len(df_sample)}개')
print(df_sample['stratum'].value_counts().sort_index())

추출 완료: 200개
stratum
large_high    15
large_low     15
large_mid     15
mid_high      17
mid_low       15
mid_mid       15
small_high    15
small_low     54
small_mid     39
Name: count, dtype: int64


## 9. 검증

In [17]:
pop_ratio  = df_f['stratum'].value_counts(normalize=True).sort_index() * 100
samp_ratio = df_sample['stratum'].value_counts(normalize=True).sort_index() * 100
samp_count = df_sample['stratum'].value_counts().sort_index()

print(f'=== 층별 모집단 vs 표본 비교 ===')
print(f'{"층":<14} {"모집단%":>8}  {"표본%":>7}  {"격차":>7}  {"표본n":>6}')
print('-' * 50)
for stratum in pop_ratio.index:
    p = pop_ratio[stratum]
    s = samp_ratio.get(stratum, 0)
    n = samp_count.get(stratum, 0)
    gap = s - p
    flag = ' ⚠' if abs(gap) > 10 else ''
    print(f'{stratum:<14} {p:>7.1f}%  {s:>6.1f}%  {gap:>+6.1f}%  {n:>6}{flag}')
print('-' * 50)

print(f'\n=== 체크리스트 ===')
checks = [
    ('층당 최소 하한 충족', all(samp_count >= MIN_PER_STRATUM)),
    ('총 표본 수',         len(df_sample) == TOTAL_N),
]
for label, ok in checks:
    print(f'  [{"✓" if ok else "✗"}] {label}')

print(f'\n=== 출시연도 분포 ===')
year_pop  = df_f['release_date'].dt.year.value_counts(normalize=True).sort_index() * 100
year_samp = df_sample['release_date'].dt.year.value_counts(normalize=True).sort_index() * 100
print(f'{"연도":<8} {"모집단%":>8}  {"표본%":>7}  {"격차":>7}')
print('-' * 36)
for yr in sorted(set(year_pop.index) | set(year_samp.index)):
    p = year_pop.get(yr, 0)
    s = year_samp.get(yr, 0)
    print(f'{yr:<8.0f} {p:>7.1f}%  {s:>6.1f}%  {s-p:>+6.1f}%')

=== 층별 모집단 vs 표본 비교 ===
층                  모집단%      표본%       격차     표본n
--------------------------------------------------
large_high         3.5%     7.5%    +4.0%      15
large_low          0.2%     7.5%    +7.3%      15
large_mid          0.2%     7.5%    +7.3%      15
mid_high           9.4%     8.5%    -0.9%      17
mid_low            5.0%     7.5%    +2.5%      15
mid_mid            7.2%     7.5%    +0.3%      15
small_high         1.8%     7.5%    +5.7%      15
small_low         43.3%    27.0%   -16.3%      54 ⚠
small_mid         29.5%    19.5%   -10.0%      39 ⚠
--------------------------------------------------

=== 체크리스트 ===
  [✓] 층당 최소 하한 충족
  [✓] 총 표본 수

=== 출시연도 분포 ===
연도           모집단%      표본%       격차
------------------------------------
2023        36.1%    39.0%    +2.9%
2024        43.1%    37.5%    -5.6%
2025        20.8%    23.5%    +2.7%


## 10. 저장

In [18]:
OUT_COLS = [
    'appid', 'name_store', 'release_date', 'genres',
    'owners', 'owners_lower', 'positive', 'negative',
    'total_reviews', 'price_spy', 'ccu', 'developers',
    'stratum', 'is_f2p'
]

out_path = '../../../data/processed/steam_market_stratified_sample.csv'
df_sample[OUT_COLS].to_csv(out_path, index=False)
print(f'저장 완료 → {out_path} ({len(df_sample)}개)')
df_sample[OUT_COLS].head()

저장 완료 → ../../../data/processed/steam_market_stratified_sample.csv (200개)


,appid,name_store,release_date,genres,owners,owners_lower,positive,negative,total_reviews,price_spy,ccu,developers,stratum,is_f2p
0,986130,Shadows of Doubt,2024-09-26,"['Action', 'Adventure', 'Indie', 'RPG']","500,000 .. 1,000,000",500000,12941,2487,15428,1249,286,ColePowered Games,large_high,False
1,1340480,The Cosmic Wheel Sisterhood,2023-08-16,"['Adventure', 'Indie']","200,000 .. 500,000",200000,4557,301,4858,1799,40,Deconstructeam,large_high,False
2,2716400,The Rise of the Golden Idol,2024-11-12,"['Adventure', 'Indie']","200,000 .. 500,000",200000,2825,178,3003,1999,59,Color Gray Games,large_high,False
3,1859910,Legend of Mortal,2024-06-14,"['Adventure', 'Indie', 'RPG', 'Strategy']","500,000 .. 1,000,000",500000,22624,9407,32031,1999,296,Obb Studio Inc.,large_high,False
4,2285150,The Front,2025-10-30,"['Action', 'Indie', 'Massively Multiplayer', '...","200,000 .. 500,000",200000,6643,3155,9798,1999,105,Samar Studio,large_high,False
